# 02 — Análisis Exploratorio de Datos (EDA)

Fase 2: EDA sobre la base de datos seleccionada.

## Calidad de los datos: faltantes, duplicados y consistencia

Esta sección diagnostica problemas de calidad antes de tomar decisiones de limpieza. Se revisan valores ausentes, duplicados, formatos, categorías y rangos estructuralmente imposibles. **No se imputan, eliminan ni transforman registros en esta etapa**; las acciones propuestas se reservan para el notebook de preprocesamiento.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Si las celdas de exploración anteriores ya crearon `df`, se reutiliza.
# La carga de respaldo permite ejecutar esta sección de forma independiente.
if "df" not in globals():
    rutas_candidatas = [
        Path("../data/raw/hcvdat0.csv"),
        Path("data/raw/hcvdat0.csv"),
    ]
    ruta_hcv = next((ruta for ruta in rutas_candidatas if ruta.exists()), None)
    if ruta_hcv is None:
        raise FileNotFoundError("No se encontró data/raw/hcvdat0.csv.")
    df = pd.read_csv(ruta_hcv)

# El diagnóstico se hace sobre una copia para preservar los datos cargados.
df_calidad = df.copy()
columna_id = df_calidad.columns[0]
print(f"Registros analizados: {len(df_calidad):,}")
print(f"Columnas analizadas: {df_calidad.shape[1]}")

### Valores faltantes

Primero se cuantifican los faltantes por columna y por fila. El porcentaje se calcula sobre los 615 registros para poder comparar la magnitud del problema entre variables.

In [ ]:
resumen_faltantes = pd.DataFrame({
    "cantidad_faltante": df_calidad.isna().sum(),
    "porcentaje_faltante": df_calidad.isna().mean().mul(100),
})
resumen_faltantes["porcentaje_faltante"] = (
    resumen_faltantes["porcentaje_faltante"].round(3)
)
resumen_faltantes = resumen_faltantes.sort_values(
    "cantidad_faltante", ascending=False
)
display(resumen_faltantes)

faltantes_totales = int(df_calidad.isna().sum().sum())
filas_con_faltantes = int(df_calidad.isna().any(axis=1).sum())
print(f"Celdas faltantes: {faltantes_totales}")
print(
    f"Filas con al menos un faltante: {filas_con_faltantes} "
    f"({filas_con_faltantes / len(df_calidad) * 100:.3f}%)"
)

In [ ]:
columnas_con_faltantes = resumen_faltantes.query(
    "cantidad_faltante > 0"
).index.tolist()
faltantes_por_categoria = (
    df_calidad.groupby("Category", observed=False)[columnas_con_faltantes]
    .agg(lambda serie: serie.isna().sum())
)
display(faltantes_por_categoria)

# Porcentajes dentro de cada categoría: permiten detectar concentración del problema.
porcentaje_faltantes_por_categoria = (
    df_calidad.groupby("Category", observed=False)[columnas_con_faltantes]
    .agg(lambda serie: serie.isna().mean() * 100)
    .round(2)
)
display(porcentaje_faltantes_por_categoria)

#### Interpretación e impacto de los faltantes

| Variable | Cantidad | Porcentaje | Impacto y tratamiento propuesto para una fase posterior |
|---|---:|---:|---|
| `ALP` | 18 | 2.927% | Es la variable más afectada. Los faltantes se concentran en Hepatitis (3), Fibrosis (9) y Cirrosis (6), por lo que no deben asumirse aleatorios: eliminar filas podría perjudicar especialmente a grupos clínicos pequeños. En preprocesamiento conviene estudiar imputación robusta por grupo diagnóstico y sexo, usando únicamente información del conjunto de entrenamiento si después se modela. |
| `CHOL` | 10 | 1.626% | Afecta a donantes y a categorías clínicas. El porcentaje global es bajo, pero una eliminación completa reduciría todavía más las clases minoritarias. Debe evaluarse una imputación robusta, comparando la distribución por diagnóstico y sexo. |
| `ALB` | 1 | 0.163% | Caso aislado dentro de Cirrosis. Puede imputarse posteriormente con una medida robusta del grupo o justificarse la exclusión del registro, después de revisar el resto de sus variables. |
| `ALT` | 1 | 0.163% | Caso aislado dentro de Hepatitis. Se recomienda el mismo análisis contextual antes de decidir entre imputación y exclusión. |
| `PROT` | 1 | 0.163% | Caso aislado dentro de Cirrosis. Debe revisarse junto con `ALB`, porque ambas representan proteínas y el registro podría contener más de una ausencia. |

En total hay **31 celdas ausentes distribuidas en 26 filas (4.228% de los registros)**. Veintitrés filas tienen un faltante, una fila tiene dos y dos filas tienen tres. La concentración de `ALP` en clases clínicas indica que eliminar todas las filas incompletas podría introducir sesgo y reducir información precisamente en los grupos menos representados.

### Registros duplicados

Se comprueban duplicados exactos y duplicados clínicos ignorando la primera columna, que funciona como identificador. Esta segunda comprobación evita que dos observaciones iguales pasen inadvertidas solo por tener identificadores distintos.

In [ ]:
duplicados_exactos = df_calidad.duplicated(keep=False)
columnas_sin_id = df_calidad.columns.drop(columna_id)
duplicados_sin_id = df_calidad.duplicated(
    subset=columnas_sin_id, keep=False
)
print(f"Filas involucradas en duplicados exactos: {duplicados_exactos.sum()}")
print(
    "Filas involucradas en duplicados al ignorar el identificador: "
    f"{duplicados_sin_id.sum()}"
)
if duplicados_sin_id.any():
    display(df_calidad.loc[duplicados_sin_id].sort_values(list(columnas_sin_id)))

**Interpretación:** no se detectan registros duplicados, ni al comparar todas las columnas ni al excluir el identificador. Por tanto, en esta versión de la base no se justifica eliminar observaciones por duplicidad. La verificación debe repetirse después de cualquier integración o transformación del dataset.

### Valores inconsistentes y posibles errores de formato

Se revisan el identificador, las categorías esperadas, espacios accidentales, variantes de mayúsculas/minúsculas y valores no numéricos en las variables de laboratorio.

In [ ]:
categorias_esperadas = {
    "0=Blood Donor",
    "0s=suspect Blood Donor",
    "1=Hepatitis",
    "2=Fibrosis",
    "3=Cirrhosis",
}
sexos_esperados = {"m", "f"}
categorias_observadas = set(df_calidad["Category"].dropna().unique())
sexos_observados = set(df_calidad["Sex"].dropna().unique())

resumen_consistencia = pd.Series({
    "categorias_no_reconocidas": len(categorias_observadas - categorias_esperadas),
    "sexos_no_reconocidos": len(sexos_observados - sexos_esperados),
    "identificadores_duplicados": int(df_calidad[columna_id].duplicated().sum()),
    "identificadores_faltantes": int(df_calidad[columna_id].isna().sum()),
    "identificador_secuencial_1_a_n": bool(
        df_calidad[columna_id].reset_index(drop=True).eq(
            pd.Series(range(1, len(df_calidad) + 1))
        ).all()
    ),
})
display(resumen_consistencia.to_frame("resultado"))

for columna in ["Category", "Sex"]:
    print(f"Valores y frecuencias de {columna}:")
    display(df_calidad[columna].value_counts(dropna=False).to_frame("frecuencia"))

In [ ]:
columnas_categoricas = ["Category", "Sex"]
columnas_numericas_esperadas = [
    columna for columna in df_calidad.columns
    if columna not in columnas_categoricas
]
filas_con_espacios = pd.Series(False, index=df_calidad.index)
for columna in columnas_categoricas:
    serie_texto = df_calidad[columna].astype("string")
    filas_con_espacios |= serie_texto.ne(serie_texto.str.strip()).fillna(False)

conversion_numerica = df_calidad[columnas_numericas_esperadas].apply(
    pd.to_numeric, errors="coerce"
)
errores_conversion = (
    conversion_numerica.isna() & df_calidad[columnas_numericas_esperadas].notna()
).sum()
valores_infinitos = np.isinf(conversion_numerica).sum()
display(pd.DataFrame({
    "errores_conversion_numerica": errores_conversion,
    "valores_infinitos": valores_infinitos,
}))
print(f"Filas con espacios al inicio o al final: {filas_con_espacios.sum()}")

**Interpretación:** `Category` contiene únicamente las cinco etiquetas documentadas y `Sex` únicamente `m` y `f`; no aparecen variantes por espacios o capitalización. Las columnas esperadas como numéricas no contienen texto inesperado ni infinitos, y el identificador recorre de 1 a 615 sin repeticiones.

Sí existe un problema de encabezado: la primera columna llega desde el CSV sin nombre y pandas la representa normalmente como `Unnamed: 0`. Sus valores muestran que es un identificador de fila, no una medición clínica. En el preprocesamiento debe renombrarse como `record_id` o excluirse de los modelos; mantenerla como predictor introduciría un orden artificial.

### Rangos aparentemente inválidos

En esta fase se aplican reglas conservadoras de validez estructural, no intervalos clínicos de referencia. Una medición extrema puede ser coherente con enfermedad y no debe etiquetarse como error sin contexto médico. Las reglas usadas son: edad entre 0 y 120 años y mediciones clínicas estrictamente positivas.

In [ ]:
columnas_clinicas = [
    "ALB", "ALP", "ALT", "AST", "BIL",
    "CHE", "CHOL", "CREA", "GGT", "PROT",
]
rangos_observados = df_calidad[["Age"] + columnas_clinicas].agg(
    ["min", "max"]
).T
rangos_observados["valores_no_positivos"] = (
    df_calidad[["Age"] + columnas_clinicas] <= 0
).sum()
display(rangos_observados)

edad_invalida = ~df_calidad["Age"].between(0, 120, inclusive="both")
clinicos_invalidos = (df_calidad[columnas_clinicas] <= 0).sum()
print(f"Edades fuera de 0–120 años: {edad_invalida.sum()}")
print(f"Mediciones clínicas no positivas: {clinicos_invalidos.sum()}")

**Interpretación:** no se detectan edades imposibles, valores negativos ni ceros en las mediciones clínicas disponibles. Los intervalos observados van de 19 a 77 años; varias pruebas presentan máximos elevados —por ejemplo `CREA`, `GGT`, `ALT`, `AST` y `BIL`—, pero estos valores pueden corresponder a pacientes con enfermedad hepática y no constituyen por sí solos errores. Su análisis como posibles valores atípicos, con métodos como IQR y contexto clínico, pertenece a una etapa posterior; aquí no se eliminan ni se modifican.

### Conclusión del diagnóstico de calidad

La base es consistente en estructura, categorías, formatos e identificadores, y no contiene duplicados. El principal problema de calidad son los faltantes: aunque representan una fracción pequeña del total, `ALP` está ausente de forma desproporcionada en las categorías clínicas. Por ello, el preprocesamiento posterior debe evitar una eliminación automática de filas y evaluar estrategias de imputación que preserven los grupos minoritarios. La columna identificadora sin nombre también debe tratarse explícitamente antes de cualquier modelo. Ninguna corrección se aplica todavía para conservar intactos los datos originales y separar el diagnóstico de la limpieza definitiva.